In [ ]:
# Precompute Masks Script

import os, json
import numpy as np
from PIL import Image
from tqdm import tqdm
import skimage.draw

# ── CONFIG ─────────────────────────────────────────────
DATA_ROOT = '/kaggle/input/datasets/varun000reddy/'
TRAIN_IMG = DATA_ROOT + 'training/train/image/'
TRAIN_ANN = DATA_ROOT + 'training/train/annos/'

SAVE_ROOT = '/kaggle/working/processed_dataset/'
SAVE_IMG  = SAVE_ROOT + 'images/'
SAVE_MASK = SAVE_ROOT + 'masks/'

IMG_SIZE = 512

# Label mapping (background = 0)
TOP5_CATEGORIES = {1: 1, 8: 2, 7: 3, 2: 4, 9: 5}

os.makedirs(SAVE_IMG, exist_ok=True)
os.makedirs(SAVE_MASK, exist_ok=True)

# ── PROCESS ────────────────────────────────────────────
ann_files = sorted([f for f in os.listdir(TRAIN_ANN) if f.endswith('.json')])

print("Precomputing masks...")

for fname in tqdm(ann_files):
    img_id   = fname.replace('.json', '')
    img_path = os.path.join(TRAIN_IMG, img_id + '.jpg')
    ann_path = os.path.join(TRAIN_ANN, fname)

    if not os.path.exists(img_path):
        continue

    # Load image
    img = Image.open(img_path).convert('RGB')
    orig_w, orig_h = img.size
    img_resized    = img.resize((IMG_SIZE, IMG_SIZE))

    # Save resized image
    img_resized.save(os.path.join(SAVE_IMG, img_id + '.jpg'))

    # Create empty mask
    mask = np.zeros((IMG_SIZE, IMG_SIZE), dtype=np.uint8)

    # Scale factors
    scale_x = IMG_SIZE / orig_w
    scale_y = IMG_SIZE / orig_h

    # Load annotation
    with open(ann_path) as f:
        ann = json.load(f)

    for key, item in ann.items():
        if not key.startswith('item'):
            continue

        cat_id = item.get('category_id')
        if cat_id not in TOP5_CATEGORIES:
            continue

        class_idx = TOP5_CATEGORIES[cat_id]
        segs      = item.get('segmentation', [])

        for polygon in segs:
            if len(polygon) < 6:
                continue

            px = np.array(polygon[0::2], dtype=np.float32) * scale_x
            py = np.array(polygon[1::2], dtype=np.float32) * scale_y

            px = np.clip(px, 0, IMG_SIZE - 1)
            py = np.clip(py, 0, IMG_SIZE - 1)

            rr, cc = skimage.draw.polygon(py, px, shape=(IMG_SIZE, IMG_SIZE))
            mask[rr, cc] = class_idx

    # Save mask as PNG (VERY IMPORTANT)
    Image.fromarray(mask).save(os.path.join(SAVE_MASK, img_id + '.png'))

print("Preprocessing complete!")

In [ ]:
# Uploading as kaggle dataset

import shutil, json, subprocess

UPLOAD_DIR = '/kaggle/working/unet_processed_upload/'
os.makedirs(UPLOAD_DIR, exist_ok=True)

shutil.copytree('/kaggle/working/processed_dataset/images',
                UPLOAD_DIR + '/images', dirs_exist_ok=True)
shutil.copytree('/kaggle/working/processed_dataset/masks',
                UPLOAD_DIR + '/masks', dirs_exist_ok=True)

# metadata
metadata = {
    "title": "vr-processed-segmentation-dataset",
    "id": "pankajdeopa/vr-processed-segmentation-dataset",
    "licenses": [{"name": "CC0-1.0"}]
}

with open(UPLOAD_DIR + '/dataset-metadata.json', 'w') as f:
    json.dump(metadata, f)

# upload
result = subprocess.run(
    ['kaggle', 'datasets', 'create', '-p', UPLOAD_DIR, '--dir-mode', 'zip'],
    capture_output=True, text=True
)

print(result.stdout)
print(result.stderr)